# ⚡ EXODUS GENESIS — Bootstrap One-Click

> **Notebook de naissance de l’Empire EXODUS V2**

---

### 🎯 Mode d’emploi
1. Exécuter les cellules **dans l’ordre** (1 → 7)
2. Les cellules 1-5 sont **obligatoires**
3. La cellule 6 est **optionnelle** (GPU uniquement, +3.8 GB)
4. La cellule 7 lance le **diagnostic complet**

### ⚠️ Prérequis
- Compte Google avec Google Drive
- ~1 Go d’espace Drive libre (minimum), ~5 Go (complet avec GPU)
- Optionnel : runtime GPU pour DepthAnything + SAM

In [ ]:
#@title 🔗 CELLULE 1 — Montage Google Drive
#@markdown Monte votre Google Drive pour stocker la structure EXODUS.

from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/DRIVE_EXODUS_V2"  #@param {type:"string"}

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

print("✅ Drive monté")
print(f"📂 Racine EXODUS : {DRIVE_ROOT}")

In [ ]:
#@title 🏗️ CELLULE 2 — Création de la Structure Sacrée
#@markdown Génère les 7 frégates + EXODUS_AI_MODELS sur votre Drive.

import subprocess, sys

REPO_URL = "https://github.com/kioka8877-ux/EXODUS-V2.git"
REPO_LOCAL = "/content/EXODUS-V2"

if not os.path.exists(REPO_LOCAL):
    print("📥 Clonage du repo EXODUS-V2...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_LOCAL], check=True)
    print("✅ Repo cloné")
else:
    print("✅ Repo déjà présent")

print("\n🏗️ Création de l'Architecture Sacrée...")
result = subprocess.run(
    [sys.executable, f"{REPO_LOCAL}/EXO_GENESIS_DRIVE.py", "--drive-root", DRIVE_ROOT, "--verbose"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(f"⚠️ Erreurs : {result.stderr}")
else:
    print("✅ Structure créée avec succès")

In [ ]:
#@title 📦 CELLULE 3 — Déploiement du Code sur le Drive
#@markdown Copie les scripts Python et notebooks depuis le repo vers les dossiers CODEBASE/ de chaque frégate.

import shutil
from pathlib import Path

REPO_LOCAL = "/content/EXODUS-V2"
units = [
    "00_CORTEX_HQ", "01_ANIMATION_ENGINE", "02_LOGISTICS_DEPOT",
    "03_SCENOGRAPHY_DOCK", "04_PHOTOGRAPHY_WING", "05_ALCHEMIST_LAB",
    "06_AIRCRAFT_CARRIER"
]

deployed = 0
for unit in units:
    src = Path(REPO_LOCAL) / unit / "CODEBASE"
    dst = Path(DRIVE_ROOT) / unit / "CODEBASE"

    if not src.exists():
        print(f"  ⏭️ {unit}/CODEBASE — pas de source, skip")
        continue

    dst.mkdir(parents=True, exist_ok=True)

    count = 0
    for f in src.iterdir():
        if f.is_file():
            shutil.copy2(str(f), str(dst / f.name))
            count += 1

    deployed += count
    print(f"  ✅ {unit}/CODEBASE — {count} fichier(s) déployé(s)")

root_scripts = ["EXO_GENESIS_DRIVE.py", "EXO_SETUP_MODELS.py", "EXO_MARSHAL.py", "phantom_link.py"]
for script in root_scripts:
    src_file = Path(REPO_LOCAL) / script
    if src_file.exists():
        shutil.copy2(str(src_file), f"{DRIVE_ROOT}/{script}")
        deployed += 1
        print(f"  ✅ {script} → racine Drive")

print(f"\n✅ Déploiement terminé — {deployed} fichier(s) au total")

In [ ]:
#@title 🎨 CELLULE 4 — Téléchargement Blender 4.0 Portable
#@markdown Télécharge Blender 4.0 directement sur le Drive (~300 MB).
#@markdown ⚡ Cloud → Drive (zéro transit mobile)

import subprocess, os

BLENDER_URL = "https://download.blender.org/release/Blender4.0/blender-4.0.0-linux-x64.tar.xz"
BLENDER_DIR = f"{DRIVE_ROOT}/EXODUS_AI_MODELS"
BLENDER_ARCHIVE = f"{BLENDER_DIR}/blender-4.0.0-linux-x64.tar.xz"
BLENDER_EXTRACTED = f"{BLENDER_DIR}/blender-4.0.0-linux-x64"

os.makedirs(BLENDER_DIR, exist_ok=True)

if os.path.exists(f"{BLENDER_EXTRACTED}/blender"):
    print("✅ Blender 4.0 déjà installé sur le Drive")
    result = subprocess.run([f"{BLENDER_EXTRACTED}/blender", "--version"], capture_output=True, text=True)
    print(f"   Version : {result.stdout.strip().splitlines()[0] if result.stdout else 'N/A'}")
else:
    print("📥 Téléchargement de Blender 4.0 (~300 MB)...")
    print("   ⚡ Cloud → Drive directement")

    subprocess.run(["wget", "-q", "--show-progress", "-O", BLENDER_ARCHIVE, BLENDER_URL], check=True)

    print("📦 Extraction...")
    subprocess.run(["tar", "-xf", BLENDER_ARCHIVE, "-C", BLENDER_DIR], check=True)

    os.remove(BLENDER_ARCHIVE)

    result = subprocess.run([f"{BLENDER_EXTRACTED}/blender", "--version"], capture_output=True, text=True)
    print(f"✅ Blender installé : {result.stdout.strip().splitlines()[0] if result.stdout else 'OK'}")

BLENDER_PATH = f"{BLENDER_EXTRACTED}/blender"
print(f"📍 Chemin : {BLENDER_PATH}")

In [ ]:
#@title 🤖 CELLULE 5 — Modèles IA Standard (~120 MB)
#@markdown Télécharge RIFE, MCprep, HDRi, Real-ESRGAN, Rhubarb.
#@markdown ⚡ Tout va directement sur le Drive.

import subprocess, os

MODELS_DIR = f"{DRIVE_ROOT}/EXODUS_AI_MODELS"

STANDARD_ASSETS = {
    "RIFE/flownet.pkl": {
        "url": "https://huggingface.co/jbilcke-hf/varnish/resolve/main/rife/flownet.pkl",
        "min_mb": 10,
        "desc": "RIFE FlowNet (interpolation frames)"
    },
    "RIFE/rife46.pkl": {
        "url": "https://huggingface.co/camenduru/FILM/resolve/main/rife/rife46.pkl",
        "min_mb": 10,
        "desc": "RIFE v4.6 (alternative)"
    },
    "McPrep/MCprep_addon.zip": {
        "url": "https://github.com/Moo-Ack-Productions/MCprep/releases/download/3.6.2/MCprep_addon_3.6.2.zip",
        "min_mb": 0.5,
        "desc": "MCprep addon Blender"
    },
    "HDRi/studio_small_09_1k.exr": {
        "url": "https://dl.polyhaven.org/file/ph-assets/HDRIs/exr/1k/studio_small_09_1k.exr",
        "min_mb": 1,
        "desc": "HDRi Studio 1K"
    },
    "HDRi/studio_small_09_2k.exr": {
        "url": "https://dl.polyhaven.org/file/ph-assets/HDRIs/exr/2k/studio_small_09_2k.exr",
        "min_mb": 3,
        "desc": "HDRi Studio 2K"
    },
    "HDRi/studio_small_09_4k.exr": {
        "url": "https://dl.polyhaven.org/file/ph-assets/HDRIs/exr/4k/studio_small_09_4k.exr",
        "min_mb": 10,
        "desc": "HDRi Studio 4K"
    },
    "REALESRGAN/realesr-general-x4v3.pth": {
        "url": "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth",
        "min_mb": 50,
        "desc": "Real-ESRGAN x4 (upscale)"
    }
}

# NOTE: Rhubarb lip-sync n'a pas de téléchargement direct fiable
# L'utilisateur devra le télécharger manuellement si nécessaire

downloaded = 0
skipped = 0

for rel_path, info in STANDARD_ASSETS.items():
    dest = f"{MODELS_DIR}/{rel_path}"
    dest_dir = os.path.dirname(dest)
    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(dest):
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        if size_mb >= info["min_mb"]:
            print(f"  ✅ {info['desc']} — déjà présent ({size_mb:.1f} MB)")
            skipped += 1
            continue

    print(f"  📥 {info['desc']}...")
    try:
        subprocess.run(["wget", "-q", "--show-progress", "-O", dest, info["url"]], check=True, timeout=300)
        size_mb = os.path.getsize(dest) / (1024 * 1024)
        print(f"     ✅ OK ({size_mb:.1f} MB)")
        downloaded += 1
    except Exception as e:
        print(f"     ⚠️ Échec : {e}")

print(f"\n✅ Modèles standard : {downloaded} téléchargé(s), {skipped} déjà présent(s)")

In [ ]:
#@title 🎮 CELLULE 6 — [OPTIONNEL GPU] DepthAnything V2 + SAM (~3.8 GB)
#@markdown ⚠️ Nécessite un runtime GPU (T4/A100).
#@markdown Ces modèles sont utilisés UNIQUEMENT par U03 (Scenography Dock).
#@markdown Si vous n’utilisez pas U03, vous pouvez IGNORER cette cellule.
#@markdown ⚡ Cloud → Drive directement (pas de transit mobile).

INSTALLER_GPU = True  #@param {type:"boolean"}

if not INSTALLER_GPU:
    print("⏭️ Installation GPU ignorée (case décochée)")
else:
    import subprocess, os

    try:
        gpu_check = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                                   capture_output=True, text=True, timeout=10)
        gpu_name = gpu_check.stdout.strip()
        if gpu_name:
            print(f"🎮 GPU détecté : {gpu_name}")
        else:
            print("⚠️ Aucun GPU détecté. Changez le runtime Colab : Runtime → Change runtime type → GPU")
            print("   Continuez quand même ? Les modèles seront téléchargés pour usage futur.")
    except:
        print("⚠️ nvidia-smi non disponible. Vérifiez le runtime GPU.")

    MODELS_DIR = f"{DRIVE_ROOT}/EXODUS_AI_MODELS"

    GPU_ASSETS = {
        "depth_anything_v2": {
            "desc": "DepthAnything V2 ViT-L (~1.4 GB)",
            "dest": f"{MODELS_DIR}/DepthAnything/depth_anything_v2_vitl.pth",
            "url": "https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth",
            "min_mb": 1000
        },
        "sam_vit_h": {
            "desc": "SAM ViT-H (~2.4 GB)",
            "dest": f"{MODELS_DIR}/SAM/sam_vit_h_4b8939.pth",
            "url": "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth",
            "min_mb": 2000
        }
    }

    for key, info in GPU_ASSETS.items():
        dest = info["dest"]
        dest_dir = os.path.dirname(dest)
        os.makedirs(dest_dir, exist_ok=True)

        if os.path.exists(dest):
            size_mb = os.path.getsize(dest) / (1024 * 1024)
            if size_mb >= info["min_mb"]:
                print(f"  ✅ {info['desc']} — déjà présent ({size_mb:.0f} MB)")
                continue

        print(f"  📥 {info['desc']}...")
        try:
            subprocess.run(["wget", "-q", "--show-progress", "-O", dest, info["url"]],
                         check=True, timeout=1800)
            size_mb = os.path.getsize(dest) / (1024 * 1024)
            print(f"     ✅ OK ({size_mb:.0f} MB)")
        except Exception as e:
            print(f"     ⚠️ Échec : {e}")

    print("\n✅ Modèles GPU installés")

In [ ]:
#@title 🩺 CELLULE 7 — Diagnostic Complet
#@markdown Vérifie l'état de toute l'installation EXODUS.

import os, json
from pathlib import Path

print("╔════════════════════════════════════════════════════════════╗")
print("║              EXODUS GENESIS — DIAGNOSTIC COMPLET            ║")
print("╚════════════════════════════════════════════════════════════╝")
print()

# --- 1. Structure des frégates ---
print("🏗️ STRUCTURE DES FRÉGATES")
print("─" * 50)
units = {
    "00_CORTEX_HQ": "Cerveau — Analyse vidéo",
    "01_ANIMATION_ENGINE": "Âme — MoCap + Facial",
    "02_LOGISTICS_DEPOT": "Arsenal — Props",
    "03_SCENOGRAPHY_DOCK": "Monde — Décors PBR",
    "04_PHOTOGRAPHY_WING": "Capture — Caméra",
    "05_ALCHEMIST_LAB": "Alchimie — Post-prod",
    "06_AIRCRAFT_CARRIER": "Porte-Avions — Export final"
}

for unit_dir, desc in units.items():
    path = Path(DRIVE_ROOT) / unit_dir
    codebase = path / "CODEBASE"
    if path.exists():
        code_count = len(list(codebase.glob("*"))) if codebase.exists() else 0
        print(f"  ✅ {unit_dir} — {desc} ({code_count} fichiers code)")
    else:
        print(f"  ❌ {unit_dir} — MANQUANT")

# --- 2. Blender ---
print(f"\n🎨 BLENDER")
print("─" * 50)
blender_path = f"{DRIVE_ROOT}/EXODUS_AI_MODELS/blender-4.0.0-linux-x64/blender"
if os.path.exists(blender_path):
    import subprocess
    result = subprocess.run([blender_path, "--version"], capture_output=True, text=True)
    version = result.stdout.strip().splitlines()[0] if result.stdout else "?"
    print(f"  ✅ {version}")
    print(f"     📍 {blender_path}")
else:
    print(f"  ❌ Blender non trouvé")
    print(f"     Attendu : {blender_path}")

# --- 3. Modèles IA ---
print(f"\n🤖 MODÈLES IA")
print("─" * 50)
models_checks = {
    "RIFE/flownet.pkl": ("RIFE FlowNet", 10),
    "RIFE/rife46.pkl": ("RIFE v4.6", 10),
    "McPrep/MCprep_addon.zip": ("MCprep Addon", 0.5),
    "HDRi/studio_small_09_1k.exr": ("HDRi 1K", 1),
    "HDRi/studio_small_09_2k.exr": ("HDRi 2K", 3),
    "HDRi/studio_small_09_4k.exr": ("HDRi 4K", 10),
    "REALESRGAN/realesr-general-x4v3.pth": ("Real-ESRGAN x4", 50),
    "DepthAnything/depth_anything_v2_vitl.pth": ("DepthAnything V2", 1000),
    "SAM/sam_vit_h_4b8939.pth": ("SAM ViT-H", 2000),
}

present = 0
total = len(models_checks)
total_size = 0

for rel_path, (name, min_mb) in models_checks.items():
    full_path = f"{DRIVE_ROOT}/EXODUS_AI_MODELS/{rel_path}"
    if os.path.exists(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        total_size += size_mb
        if size_mb >= min_mb:
            print(f"  ✅ {name} ({size_mb:.1f} MB)")
            present += 1
        else:
            print(f"  ⚠️ {name} — fichier trop petit ({size_mb:.1f} MB < {min_mb} MB)")
    else:
        gpu_only = min_mb >= 1000
        tag = " [GPU optionnel]" if gpu_only else ""
        print(f"  ❌ {name} — manquant{tag}")

# --- 4. Scripts racine ---
print(f"\n📜 SCRIPTS RACINE")
print("─" * 50)
root_scripts = ["EXO_GENESIS_DRIVE.py", "EXO_SETUP_MODELS.py", "EXO_MARSHAL.py", "phantom_link.py"]
for script in root_scripts:
    spath = f"{DRIVE_ROOT}/{script}"
    if os.path.exists(spath):
        print(f"  ✅ {script}")
    else:
        print(f"  ❌ {script} — manquant")

# --- 5. Résumé ---
print(f"\n{'\u2550' * 50}")
print(f"📊 RÉSUMÉ")
print(f"{'\u2550' * 50}")
print(f"  Modèles IA : {present}/{total} présents ({total_size:.0f} MB total)")
blender_ok = os.path.exists(blender_path)
print(f"  Blender    : {'✅ Opérationnel' if blender_ok else '❌ Manquant'}")

if blender_ok and present >= 7:
    print(f"\n🎉 EXODUS EST OPÉRATIONNEL — L'Empire peut commencer la production !")
elif blender_ok:
    print(f"\n⚡ EXODUS EST FONCTIONNEL — Blender OK, certains modèles optionnels manquent")
else:
    print(f"\n⚠️ EXODUS INCOMPLET — Relancez les cellules en erreur")